# KEYSTONE Cowork — PinkCherry v1.8 Hybrid (BF16) + MiniMax H3 Ref2VA & Qwen 3.8 Runtime
**Target Runtimes:** Google Colab Pro (`1x A100-SXM4-80GB`, `2x L4`, `1x L4`) & Multi-GPU Cloud Workstations

### Architecture & Capabilities
- **Diffusion Backbone (`UNETLoader` & `CheckpointLoaderSimple`)**: `SexGod1979/PinkCherry-NSFW_MiniMax-H3` -> `1.8-hybrid/PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors`
- **Omni-Modal Text/Vision Encoder (`CLIPLoader`)**: `Comfy-Org/MiniMax-H3` -> `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`
- **Dual Audio-Video VAEs (`VAELoader`)**: `minimax_h3_video_vae_fp16.safetensors` + `minimax_h3_audio_vae_fp32.safetensors`
- **Turbo LoRAs (`LoraLoaderModelOnly`)**: `MiniMaxh3 Turbo/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors`
- **Cognition Engine**: `huihui-ai/Huihui-Qwen3.8-27B-abliterated` (`bfloat16`)

In [ ]:
# ============================================================================
# CELL 1: GPU Topology Auto-Discovery & Credential Resolution
# ============================================================================
import os
import pathlib
import subprocess
import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def resolve_secret(name: str, default: str = "") -> str:
    val = os.environ.get(name, "").strip()
    if val:
        return val
    try:
        from google.colab import userdata as colab_userdata
        val = (colab_userdata.get(name) or "").strip()
        if val:
            os.environ[name] = val
            return val
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        val = (UserSecretsClient().get_secret(name) or "").strip()
        if val:
            os.environ[name] = val
            return val
    except Exception:
        pass
    return default

HF_TOKEN = resolve_secret("HF_TOKEN")
assert torch.cuda.is_available(), "FATAL: CUDA GPU runtime is required."

GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
GPU_VRAM_GB = [round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2) for i in range(GPU_COUNT)]
TOTAL_VRAM_GB = round(sum(GPU_VRAM_GB), 2)

# Dynamic tensor parallel & VRAM flags across A100-80GB, Dual-L4 (2x24GB), Single-L4, Dual-T4
TENSOR_PARALLEL_SIZE = 2 if GPU_COUNT >= 2 else 1
COMFY_VRAM_FLAG = "--normalvram"  # Allows Qwen 3.8 reasoning model & PinkCherry BF16 to coexist in VRAM
CPU_OFFLOAD_GB = 0 if TOTAL_VRAM_GB >= 60 else (16 if TOTAL_VRAM_GB >= 40 else 28)

print(f"[KEYSTONE] Detected {GPU_COUNT}x GPU: {GPU_NAMES} | VRAM per GPU: {GPU_VRAM_GB} GiB (Total: {TOTAL_VRAM_GB} GiB)")
print(f"[KEYSTONE] ComfyUI Mode: {COMFY_VRAM_FLAG} | Cognition TP={TENSOR_PARALLEL_SIZE}, CPU_OFFLOAD_GB={CPU_OFFLOAD_GB}")

In [ ]:
# ============================================================================
# CELL 2: Detect/Install ComfyUI + KJNodes + VideoHelperSuite + Cloudflared
# ============================================================================
import os
import pathlib
import subprocess
import sys

# Auto-detect existing ComfyUI installation (/content/ComfyUI or /tmp/ComfyUI)
CANDIDATE_DIRS = [pathlib.Path("/content/ComfyUI"), pathlib.Path("/tmp/ComfyUI"), pathlib.Path.cwd() / "ComfyUI"]
COMFY_DIR = next((p for p in CANDIDATE_DIRS if (p / "main.py").exists()), pathlib.Path("/content/ComfyUI"))

if not (COMFY_DIR / "main.py").exists():
    print(f"[KEYSTONE] Cloning ComfyUI into {COMFY_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_transfer]", "gguf", "sageattention", " librosa", "soundfile"], check=False)

CUSTOM_NODES = {
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
}

custom_nodes_root = COMFY_DIR / "custom_nodes"
custom_nodes_root.mkdir(parents=True, exist_ok=True)

for name, repo_url in CUSTOM_NODES.items():
    dest = custom_nodes_root / name
    if not dest.exists():
        print(f"[KEYSTONE] Installing custom node pack: {name}...")
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(dest)], check=True)
    req_file = dest / "requirements.txt"
    if req_file.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=False)

CLOUDFLARED_BIN = pathlib.Path("/usr/local/bin/cloudflared")
if not CLOUDFLARED_BIN.exists():
    subprocess.run([
        "wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(CLOUDFLARED_BIN)
    ], check=True)
    CLOUDFLARED_BIN.chmod(0o755)

print(f"[KEYSTONE] ComfyUI root ready at: {COMFY_DIR}")

In [ ]:
# ============================================================================
# CELL 3: Stage PinkCherry v1.8 Hybrid BF16 + Official MiniMax H3 Companion Weights
# ============================================================================
import os
import pathlib
from huggingface_hub import hf_hub_download

PINKCHERRY_REPO = "SexGod1979/PinkCherry-NSFW_MiniMax-H3"
PINKCHERRY_BF16_FILE = "PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors"
PINKCHERRY_SUBPATH = f"1.8-hybrid/{PINKCHERRY_BF16_FILE}"
MINIMAX_REPO = "Comfy-Org/MiniMax-H3"

ckpt_sub_dir = COMFY_DIR / "models" / "checkpoints" / "1.8-hybrid"
diff_dir = COMFY_DIR / "models" / "diffusion_models"
diff_sub_dir = diff_dir / "1.8-hybrid"
te_dir = COMFY_DIR / "models" / "text_encoders"
vae_dir = COMFY_DIR / "models" / "vae"
lora_dir = COMFY_DIR / "models" / "loras" / "MiniMaxh3 Turbo"

for d in (ckpt_sub_dir, diff_dir, diff_sub_dir, te_dir, vae_dir, lora_dir):
    d.mkdir(parents=True, exist_ok=True)

# 1. Locate or download PinkCherry_hybrid_v1.8_minimax_h3_pruned_bf16.safetensors
existing_candidates = [
    ckpt_sub_dir / PINKCHERRY_BF16_FILE,
    diff_dir / PINKCHERRY_BF16_FILE,
    diff_sub_dir / PINKCHERRY_BF16_FILE,
]
source_bf16 = next((p for p in existing_candidates if p.exists() and p.stat().st_size > 1_000_000_000), None)

if source_bf16 is None:
    print(f"[KEYSTONE] Downloading {PINKCHERRY_SUBPATH} from {PINKCHERRY_REPO}...")
    dl_path = hf_hub_download(
        repo_id=PINKCHERRY_REPO,
        filename=PINKCHERRY_SUBPATH,
        local_dir=str(COMFY_DIR / "models" / "checkpoints"),
        token=HF_TOKEN or None,
    )
    source_bf16 = pathlib.Path(dl_path).resolve()
else:
    print(f"[KEYSTONE] Found existing PinkCherry BF16 checkpoint at: {source_bf16}")

# Symlink into both diffusion_models/ (for UNETLoader) and checkpoints/1.8-hybrid/ (for CheckpointLoaderSimple)
for target_link in [diff_dir / PINKCHERRY_BF16_FILE, diff_sub_dir / PINKCHERRY_BF16_FILE, ckpt_sub_dir / PINKCHERRY_BF16_FILE]:
    if not target_link.exists():
        try:
            target_link.symlink_to(source_bf16)
            print(f"[KEYSTONE] Linked {target_link} -> {source_bf16}")
        except Exception as exc:
            print(f"[KEYSTONE] Link note for {target_link}: {exc}")

# 2. Stage Official Comfy-Org/MiniMax-H3 Companion Assets (Text Encoder, Video/Audio VAEs, Turbo LoRAs)
COMPANION_ASSETS = [
    ("split_files/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", te_dir / "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("split_files/vae/minimax_h3_video_vae_fp16.safetensors", vae_dir / "minimax_h3_video_vae_fp16.safetensors"),
    ("split_files/vae/minimax_h3_audio_vae_fp32.safetensors", vae_dir / "minimax_h3_audio_vae_fp32.safetensors"),
    ("split_files/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", lora_dir / "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"),
    ("split_files/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", lora_dir / "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"),
]

stage_cache = COMFY_DIR / "models" / "_hf_stage"
stage_cache.mkdir(parents=True, exist_ok=True)

for remote_rel, final_dest in COMPANION_ASSETS:
    if final_dest.exists() and final_dest.stat().st_size > 1_000_000:
        print(f"[KEYSTONE] Verified companion asset: {final_dest.name} ({round(final_dest.stat().st_size / (1024**2), 1)} MB)")
        continue
    print(f"[KEYSTONE] Downloading {remote_rel} from {MINIMAX_REPO}...")
    downloaded = pathlib.Path(hf_hub_download(
        repo_id=MINIMAX_REPO,
        filename=remote_rel,
        local_dir=str(stage_cache),
        token=HF_TOKEN or None,
    )).resolve()
    if final_dest.exists() or final_dest.is_symlink():
        final_dest.unlink()
    final_dest.symlink_to(downloaded)
    print(f"[KEYSTONE] Staged {final_dest.name}")

print("[KEYSTONE] All PinkCherry v1.8 BF16 & MiniMax H3 weights staged!")

In [ ]:
# ============================================================================
# CELL 4: Launch / Reload ComfyUI (Port 8188) & Cloudflare Tunnel
# ============================================================================
import json
import re
import subprocess
import sys
import time
import urllib.request

COMFY_PORT = 8188

# Restart ComfyUI cleanly so newly linked models & custom nodes are registered
subprocess.run(["pkill", "-f", "main.py.*8188"], check=False)
time.sleep(2)

comfy_log = open("/tmp/comfyui_pinkcherry.log", "w")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT), COMFY_VRAM_FLAG],
    cwd=str(COMFY_DIR),
    stdout=comfy_log,
    stderr=subprocess.STDOUT,
)

print("[KEYSTONE] Waiting for ComfyUI on port 8188...")
for _ in range(90):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2) as resp:
            if resp.status == 200:
                print("[KEYSTONE] ComfyUI is ONLINE!")
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI failed to start within 180s. Check /tmp/comfyui_pinkcherry.log")

# Verify registered loaders
for loader, field in [("UNETLoader", "unet_name"), ("CLIPLoader", "clip_name"), ("VAELoader", "vae_name"), ("LoraLoaderModelOnly", "lora_name")]:
    with urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/object_info/{loader}", timeout=10) as r:
        data = json.load(r)
        choices = data.get(loader, {}).get("input", {}).get("required", {}).get(field, [[]])[0]
        print(f"  -> {loader}.{field}: {choices}")

# Start Cloudflare Tunnel if not already running
tunnel_log_path = "/tmp/cloudflared_comfy.log"
PUBLIC_COMFY_URL = None
if os.path.exists(tunnel_log_path):
    txt = open(tunnel_log_path).read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
    if m:
        PUBLIC_COMFY_URL = m.group(0)

if not PUBLIC_COMFY_URL:
    t_log = open(tunnel_log_path, "w")
    subprocess.Popen(
        [str(CLOUDFLARED_BIN), "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}", "--no-autoupdate"],
        stdout=t_log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        time.sleep(1)
        txt = open(tunnel_log_path).read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m:
            PUBLIC_COMFY_URL = m.group(0)
            break

print("=" * 72)
print(f"KEYSTONE COMFY_BASE_URL = {PUBLIC_COMFY_URL}")
print("=" * 72)

In [ ]:
# ============================================================================
# CELL 5: KEYSTONE Embedded Cognition Server (Huihui-Qwen3.8-27B-abliterated)
# ============================================================================
LAUNCH_COGNITION_SERVER = True
COGNITION_MODEL_ID = "huihui-ai/Huihui-Qwen3.8-27B-abliterated"
SERVED_ALIAS = "huihui-qwen3.8-abliterated"
OPENAI_API_KEY = "keystone-live-key"

if LAUNCH_COGNITION_SERVER:
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", COGNITION_MODEL_ID,
        "--served-model-name", COGNITION_MODEL_ID, SERVED_ALIAS,
        "--api-key", OPENAI_API_KEY,
        "--dtype", "bfloat16",
        "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
        "--gpu-memory-utilization", "0.48",
        "--cpu-offload-gb", str(CPU_OFFLOAD_GB),
        "--max-model-len", "4096",
        "--trust-remote-code",
        "--port", "8000",
    ]
    print("[KEYSTONE] Launching Cognition Server:", " ".join(cmd))
    subprocess.Popen(cmd, stdout=open("/tmp/vllm_qwen38.log", "w"), stderr=subprocess.STDOUT)